In [14]:
import pandas as pd
import numpy as np
import re
import ast

Vì embedding model không ăn trực tiếp bảng Excel kiểu nhiều cột lộn xộn. Nó cần một đoạn text rõ ràng để biến thành vector. biến 

profile thành text chuẩn để embedding

In [15]:
def is_null(x):
    if x is None:
        return True
    if isinstance(x, float) and pd.isna(x):
        return True
    try:
        return pd.isna(x)
    except:
        return False


def clean_text_basic(text):
    if is_null(text):
        return ""
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def try_parse_list(x):
    """
    Parse field có thể là:
    - list thật
    - string kiểu "['a', 'b']"
    - string kiểu 'a, b, c'
    """
    if is_null(x):
        return []

    if isinstance(x, list):
        return x

    if isinstance(x, (tuple, set)):
        return list(x)

    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []

        # thử parse kiểu Python list
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    return list(parsed)
            except:
                pass

        # fallback: split dấu phẩy
        if "," in s:
            return [item.strip() for item in s.split(",") if item.strip()]

        return [s]

    return [x]


def normalize_list_field(x):
    """
    - parse về list
    - clean text
    - bỏ rỗng
    - bỏ trùng, giữ thứ tự
    """
    items = try_parse_list(x)
    cleaned = []
    seen = set()

    for item in items:
        val = clean_text_basic(item)
        if not val:
            continue
        key = val.lower()
        if key not in seen:
            seen.add(key)
            cleaned.append(val)

    return cleaned


def join_list_field(x, sep=", "):
    return sep.join(normalize_list_field(x))

In [16]:
candidate_df = pd.read_excel("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step05_candidate_profile/05_candidate_profiles.xlsx")
job_df = pd.read_excel("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step06_job_and_matching/06_job_profiles.xlsx")

In [17]:
print("Candidate columns:")
print(candidate_df.columns.tolist())

print("\nJob columns:")
print(job_df.columns.tolist())

Candidate columns:
['candidate_id', 'profile_text_clean', 'skills_raw_detected', 'mapped_skills', 'skill_groups', 'skill_subgroups', 'n_raw_skills', 'n_mapped_skills', 'dominant_group', 'section_summary', 'section_experience', 'section_projects', 'section_other']

Job columns:
['job_id', 'job_title', 'job_text_clean', 'job_skills_detected', 'mapped_skills', 'skill_groups', 'skill_subgroups', 'n_job_skills_detected', 'n_mapped_skills', 'dominant_group']


In [18]:
def build_candidate_text(row):
    parts = []

    dominant_group = clean_text_basic(row.get("dominant_group"))
    mapped_skills = join_list_field(row.get("mapped_skills"))
    skill_groups = join_list_field(row.get("skill_groups"))
    skill_subgroups = join_list_field(row.get("skill_subgroups"))

    section_summary = clean_text_basic(row.get("section_summary"))
    section_experience = clean_text_basic(row.get("section_experience"))
    section_projects = clean_text_basic(row.get("section_projects"))
    section_other = clean_text_basic(row.get("section_other"))

    if dominant_group:
        parts.append(f"Candidate group: {dominant_group}")
    if mapped_skills:
        parts.append(f"Standardized skills: {mapped_skills}")
    if skill_groups:
        parts.append(f"Skill groups: {skill_groups}")
    if skill_subgroups:
        parts.append(f"Skill subgroups: {skill_subgroups}")
    if section_summary:
        parts.append(f"Summary: {section_summary}")
    if section_experience:
        parts.append(f"Experience: {section_experience}")
    if section_projects:
        parts.append(f"Projects: {section_projects}")
    if section_other:
        parts.append(f"Other: {section_other}")

    text = ". ".join(parts).strip()

    if text:
        text = "query: " + text

    return text

In [19]:
def build_job_text(row):
    parts = []

    job_title = clean_text_basic(row.get("job_title"))
    dominant_group = clean_text_basic(row.get("dominant_group"))
    mapped_skills = join_list_field(row.get("mapped_skills"))
    skill_groups = join_list_field(row.get("skill_groups"))
    skill_subgroups = join_list_field(row.get("skill_subgroups"))
    job_text_clean = clean_text_basic(row.get("job_text_clean"))

    if job_title:
        parts.append(f"Job title: {job_title}")
    if dominant_group:
        parts.append(f"Job group: {dominant_group}")
    if mapped_skills:
        parts.append(f"Standardized skills: {mapped_skills}")
    if skill_groups:
        parts.append(f"Skill groups: {skill_groups}")
    if skill_subgroups:
        parts.append(f"Skill subgroups: {skill_subgroups}")
    if job_text_clean:
        parts.append(f"Job description: {job_text_clean}")

    text = ". ".join(parts).strip()

    if text:
        text = "passage: " + text

    return text

In [20]:
candidate_df["candidate_text"] = candidate_df.apply(build_candidate_text, axis=1)
job_df["job_text"] = job_df.apply(build_job_text, axis=1)

In [21]:
candidate_df[["candidate_id", "candidate_text"]].head(5)

,candidate_id,candidate_text
0,C001,query: Candidate group: Software Development. ...
1,C002,query: Candidate group: Software Development. ...
2,C003,query: Candidate group: Data & Databases. Stan...
3,C004,query: Candidate group: IT Infrastructure & Op...
4,C005,query: Other: qa engineer experienced in manua...


In [22]:
job_df[["job_id", "job_text"]].head(5)

,job_id,job_text
0,J001,passage: Job title: Angular Frontend Engineer....
1,J002,passage: Job title: Next.js Web Developer. Job...
2,J003,passage: Job title: Frontend UI Engineer. Job ...
3,J004,passage: Job title: Vue Frontend Developer. Jo...
4,J005,passage: Job title: React Frontend Developer. ...


In [23]:
candidate_df["candidate_text_len"] = candidate_df["candidate_text"].str.len()
job_df["job_text_len"] = job_df["job_text"].str.len()

print(candidate_df["candidate_text_len"].describe())
print(job_df["job_text_len"].describe())

count     20.000000
mean     323.600000
std       69.022803
min      176.000000
25%      312.750000
50%      334.000000
75%      365.750000
max      419.000000
Name: candidate_text_len, dtype: float64
count     80.000000
mean     301.437500
std       81.353245
min      179.000000
25%      203.500000
50%      324.500000
75%      358.750000
max      448.000000
Name: job_text_len, dtype: float64


In [11]:
candidate_df[candidate_df["candidate_text_len"] < 50][["candidate_id", "candidate_text"]].head(20)

,candidate_id,candidate_text


In [24]:
job_df[job_df["job_text_len"] < 50][["job_id", "job_text"]].head(20)

,job_id,job_text


In [25]:
candidate_text_df = candidate_df[["candidate_id", "candidate_text"]].copy()
job_text_df = job_df[["job_id", "job_text"]].copy()

candidate_text_df.to_excel("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/08_candidate_texts.xlsx", index=False)
job_text_df.to_excel("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/09_job_texts.xlsx", index=False)